In [ ]:
%run kaggle/setup.py

In [ ]:
# Verify data structure
import os
print("Input directory structure:")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
# Run preprocessing
import subprocess
import sys

# Ensure cache directory exists
os.makedirs('/kaggle/working/cache', exist_ok=True)

# Run preprocess_data.py
cmd = [
    sys.executable, 'scripts/preprocess_data.py',
    '--input_dir', '/kaggle/input/sonicsight-data',
    '--output_dir', '/kaggle/working/cache',
    '--n_sources', '4',
    '--val_ratio', '0.1',
    '--test_ratio', '0.1',
    '--seed', '42'
]
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Preprocessing failed with code {result.returncode}")

In [ ]:
# Verify cache/index.json exists and has expected keys
import json

index_path = '/kaggle/working/cache/index.json'
if os.path.exists(index_path):
    with open(index_path) as f:
        index = json.load(f)
    
    print(f"Index keys: {list(index.keys())}")
    for split in ['train', 'val', 'test']:
        if split in index:
            print(f"  {split}: {len(index[split])} clips")
            if index[split]:
                sample = index[split][0]
                print(f"    Sample keys: {list(sample.keys())}")
                print(f"    Sample clip_id: {sample.get('clip_id', 'N/A')}")
    print("\nCache verification PASSED")
else:
    raise FileNotFoundError(f"Index not found: {index_path}")

In [ ]:
# Sanity check: load 1 batch through full model
import sys
sys.path.insert(0, '/kaggle/working/SonicSightDino')

import torch
from src.models.separator import Separator
from src.data.datamodule import AudioVisualDataModule

# Load model
cfg = {'model': {'n_sources': 4}}
model = Separator(n_sources=4)
model.eval()

# Create dummy data matching expected shapes
B = 1  # batch size
mixture_stft = torch.randn(B, 2, 257, 601)  # [B, 2, F, T]
N_frames = 150
video_frames = torch.randn(B, N_frames, 3, 448, 448)  # [B, N_frames, 3, H, W]

print(f"Input shapes:")
print(f"  mixture_stft: {mixture_stft.shape}")
print(f"  video_frames: {video_frames.shape}")

# Forward pass
with torch.no_grad():
    output = model(mixture_stft, video_frames)

print(f"\nOutput shape: {output.shape}")
print(f"Expected: [B, N_sources, 2, F, T] = [1, 4, 2, 257, 601]")

assert output.shape == (B, 4, 2, 257, 601), f"Bad shape: {output.shape}"
print("\nSanity check PASSED - output shape correct!")